# Step 1

In [ ]:
%run process_data.py --dataset dataset_name/

# Step 2

In [ ]:
#Load environment
import importlib, pickle
prep = importlib.import_module('process_data') 


tf = prep.tf
np = prep.np
optimizers = prep.optimizers
plt = prep.plt
sns = prep.sns
classification_report = prep.classification_report
precision_score = prep.precision_score
recall_score = prep.recall_score
f1_score = prep.f1_score
confusion_matrix = prep.confusion_matrix


print_custom_metrics = getattr(prep, 'print_custom_metrics', None)
calculate_mcc = getattr(prep, 'calculate_mcc', None)

with open('prepare_env.pkl', 'rb') as f:
    env = pickle.load(f)

inception_model_instance = env['model']
X_train_resized = env['X_train']
y_train_encoded = env['y_train']
X_val_resized = env['X_val']
y_val_encoded = env['y_val']
X_test_resized = env['X_test']
y_test_encoded = env['y_test']
classes = env['classes']

# Step 3

In [ ]:
inception_model_instance.compile(optimizer=optimizers.Adam(),
                                 loss='binary_crossentropy',  
                                  metrics=['accuracy'])

history_inception = inception_model_instance.fit(X_train_resized, 
                                                 y_train_encoded,  
                                                 validation_data=(X_val_resized, y_val_encoded),
                                                 epochs=20,
                                                 batch_size=32)

# Step 4

In [ ]:
train_loss_inception, train_accuracy_inception = inception_model_instance.evaluate(X_train_resized, y_train_encoded)
print("\nTraining Accuracy (InceptionV3):", train_accuracy_inception)


y_train_pred_proba = inception_model_instance.predict(X_train_resized)
y_train_pred_classes = (y_train_pred_proba > 0.5).astype(int).flatten()



test_loss_inception, test_accuracy_inception = inception_model_instance.evaluate(X_test_resized, y_test_encoded)
print("\nTest Accuracy (InceptionV3):", test_accuracy_inception)

y_test_pred_proba = inception_model_instance.predict(X_test_resized)
y_test_pred_classes = (y_test_pred_proba > 0.5).astype(int).flatten()

mcc_score = calculate_mcc(y_test_encoded, y_test_pred_proba)

print(f"\nMatthews Correlation Coefficient (MCC): {mcc_score:.3f}")


test_cm_inception = confusion_matrix(y_test_encoded, y_test_pred_classes)
plt.figure(figsize=(5, 5))
sns.heatmap(test_cm_inception, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Matrice de Confusion (Test)')
plt.savefig("/path_to_save_the_file/confusion_matrix.pdf", format="pdf", bbox_inches="tight")
plt.show()

print("Classification Report (Test):")
print(classification_report(y_test_encoded, y_test_pred_classes, target_names=classes))


print("\nCustom Metrics (Test):")
print_custom_metrics(y_test_encoded, y_test_pred_classes, classes)


inception_metrics = {
    "Training Accuracy": train_accuracy_inception * 100,
    "Test Accuracy": test_accuracy_inception * 100,
    "Test MCC": mcc_score * 100, 
    "Test Precision": precision_score(y_test_encoded, y_test_pred_classes, average='weighted') * 100,
    "Test Recall": recall_score(y_test_encoded, y_test_pred_classes, average='weighted') * 100,
    "Test F1-Score": f1_score(y_test_encoded, y_test_pred_classes, average='weighted') * 100
}